## Differences between h1unfinished.py and h1.py

### h1unfinished.py (Starter File)
- Contains TODO comments with instructions
- Missing implementation code for:
  - Data shape display
  - Model summary call
  - Model compilation (optimizer, loss function)
  - Training loop (model.fit)
  - Missing the evaluation line (commented out)
- References undefined variables (`test_acc`)

### h1.py (Completed File)
- All implementation is complete
- Displays data shapes
- Shows model architecture with `model.summary()`
- Compiles model with Adam optimizer and categorical crossentropy
- Trains for 20 epochs with batch size 128 and 10% validation split
- Evaluates and prints test accuracy
- Working end-to-end pipeline

## Import Libraries

In [12]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import matplotlib.pyplot as plt
import numpy as np

ModuleNotFoundError: No module named 'tensorflow'

I start by creating a test dict of names, and next iterate through the plots to create my tensor boilderplate

In [ ]:
# Load CIFAR-10 dataset
(x_train, y_train), (x_test, y_test) = keras.datasets.cifar10.load_data()


print(f"Training images shape: {x_train.shape}")  # (50000, 32, 32, 3)
print(f"Training labels shape: {y_train.shape}")  # (50000, 1)
print(f"Test images shape: {x_test.shape}")        # (10000, 32, 32, 3)
print(f"Test labels shape: {y_test.shape}")        # (10000, 1)


class_names = ['airplane', 'automobile', 'bird', 'cat', 'deer', 
               'dog', 'frog', 'horse', 'ship', 'truck']

plt.figure(figsize=(12, 6))
for i in range(10):
    plt.subplot(2, 5, i + 1)
    plt.imshow(x_train[i])
    plt.title(class_names[y_train[i][0]])
    plt.axis('off')
plt.tight_layout()
plt.show()

Next, I persued to preprocessing data by normalizing the pixel values to [0,1] range and convert lavels to a one-hot encoding.

In [ ]:
# Normalize pixel values from [0, 255] to [0, 1]
x_train = x_train.astype("float32") / 255.0
x_test  = x_test.astype("float32") / 255.0

# Convert to one-hot encoding
num_classes = 10
y_train = keras.utils.to_categorical(y_train, num_classes)
y_test  = keras.utils.to_categorical(y_test, num_classes)

print(f"After one-hot encoding:")
print(f"y_train shape: {y_train.shape}")  # (50000, 10)
print(f"y_test shape: {y_test.shape}")    # (10000, 10)

## Data Augmentation

Apply random transformations during training to improve model generalization.

In [ ]:
# Create data augmentation pipeline
data_aug = keras.Sequential([
    layers.RandomFlip("horizontal"),          # Flip images left-right
    layers.RandomTranslation(0.1, 0.1),       # Shift images by 10%
    layers.RandomRotation(0.05),              # Rotate by ±5%
])

# Visualize augmented images
plt.figure(figsize=(12, 4))
sample_img = x_train[0:1]  # Take first image
for i in range(6):
    plt.subplot(2, 3, i + 1)
    augmented = data_aug(sample_img, training=True)
    plt.imshow(augmented[0])
    plt.title(f'Augmented {i+1}')
    plt.axis('off')
plt.tight_layout()
plt.show()

## Build CNN Architecture

The model consists of:
- **Data augmentation layer** (applied during training)
- **3 Convolutional blocks** with increasing filters (32 → 64 → 128)
- **Global Average Pooling** to reduce spatial dimensions
- **Dense layer** with dropout for classification

In [ ]:
# Build CNN using Functional API
inputs = keras.Input(shape=(32, 32, 3))
x = data_aug(inputs)

# Block 1: 32 filters
x = layers.Conv2D(32, 3, padding="same", use_bias=False)(x)
x = layers.ReLU()(x)
x = layers.Conv2D(32, 3, padding="same", use_bias=False)(x)
x = layers.ReLU()(x)

# Block 2: 64 filters
x = layers.Conv2D(64, 3, padding="same", use_bias=False)(x)
x = layers.ReLU()(x)
x = layers.Conv2D(64, 3, padding="same", use_bias=False)(x)
x = layers.ReLU()(x)

# Block 3: 128 filters
x = layers.Conv2D(128, 3, padding="same", use_bias=False)(x)
x = layers.ReLU()(x)
x = layers.Conv2D(128, 3, padding="same", use_bias=False)(x)
x = layers.ReLU()(x)

# Classification head
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(128, activation="relu")(x)
x = layers.Dropout(0.4)(x)

outputs = layers.Dense(num_classes, activation="softmax")(x)

model = keras.Model(inputs, outputs)

## Display Model Architecture

**What was added:** `model.summary()` call

In [ ]:
# === WHAT WAS ADDED ===
model.summary()

## Compile the Model

**What was added:** Complete `model.compile()` call with:
- **Optimizer:** Adam with learning rate 0.001
- **Loss:** Categorical crossentropy (for multi-class classification)
- **Metrics:** Accuracy

In [ ]:
# === WHAT WAS ADDED ===
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

print("Model compiled successfully!")

## Train the Model

**What was added:** Complete `model.fit()` call with:
- **Training data:** x_train, y_train
- **Batch size:** 128
- **Epochs:** 20
- **Validation split:** 10% of training data

In [ ]:
# === WHAT WAS ADDED ===
history = model.fit(
    x_train, y_train,
    batch_size=128,
    epochs=20,
    validation_split=0.1,
    verbose=1
)

## Training History Visualization

In [ ]:
# Plot training history
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy plot
axes[0].plot(history.history['accuracy'], label='Training Accuracy', linewidth=2)
axes[0].plot(history.history['val_accuracy'], label='Validation Accuracy', linewidth=2)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].set_title('Model Accuracy Over Time')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Loss plot
axes[1].plot(history.history['loss'], label='Training Loss', linewidth=2)
axes[1].plot(history.history['val_loss'], label='Validation Loss', linewidth=2)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].set_title('Model Loss Over Time')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Evaluate on Test Set

**What was added:** Uncommented and properly executed `model.evaluate()`

In [ ]:
# === WHAT WAS ADDED ===
test_loss, test_acc = model.evaluate(x_test, y_test, verbose=0)
print(f"Test accuracy: {test_acc:.4f}")
print(f"Test loss: {test_loss:.4f}")

## Make Predictions on Sample Images

In [ ]:
# Get predictions for first 10 test images
predictions = model.predict(x_test[:10])
predicted_classes = np.argmax(predictions, axis=1)
true_classes = np.argmax(y_test[:10], axis=1)

# Visualize predictions
plt.figure(figsize=(14, 6))
for i in range(10):
    plt.subplot(2, 5, i + 1)
    plt.imshow(x_test[i])
    
    pred_class = class_names[predicted_classes[i]]
    true_class = class_names[true_classes[i]]
    confidence = predictions[i][predicted_classes[i]] * 100
    
    color = 'green' if predicted_classes[i] == true_classes[i] else 'red'
    plt.title(f'Pred: {pred_class}\nTrue: {true_class}\n{confidence:.1f}%', 
              color=color, fontsize=9)
    plt.axis('off')
    
plt.tight_layout()
plt.show()

## Summary of What Was Added

### 1. **Data Shape Display** (Lines 13-16)
```python
print(f"Training images shape: {x_train.shape}")
print(f"Training labels shape: {y_train.shape}")
print(f"Test images shape: {x_test.shape}")
print(f"Test labels shape: {y_test.shape}")
```
Shows the dimensions of the dataset for verification.

### 2. **Model Summary** (Line 66)
```python
model.summary()
```
Displays the complete network architecture with parameter counts.

### 3. **Model Compilation** (Lines 69-73)
```python
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)
```
Configures the model for training with Adam optimizer and appropriate loss function.

### 4. **Model Training** (Lines 76-82)
```python
history = model.fit(
    x_train, y_train,
    batch_size=128,
    epochs=20,
    validation_split=0.1,
    verbose=1
)
```
Trains the model on CIFAR-10 data with specified hyperparameters.

### 5. **Test Evaluation** (Line 85)
```python
test_loss, test_acc = model.evaluate(x_test, y_test, verbose=0)
```
Evaluates the trained model on the held-out test set.

---

These additions complete the end-to-end deep learning pipeline: **Load → Preprocess → Build → Compile → Train → Evaluate**